# Previous Loans – Feature Engineering

Cleans and aggregates the `previous_application` dataset for the Home Credit default prediction pipeline.

**Outputs**
- `prev_loan_data_processed.parquet` — enriched loan-level table (used as metadata by CC, POS, Payments notebooks)
- `prev_loans_aggregate_final_processed.parquet` — customer-level aggregated features

**Pipeline overview**
1. Load data & restrict to training customers
2. Clean sentinel values (`365243`, `XNA`)
3. Derive ratio features
4. Save enriched loan-level parquet
5. Split into Approved/Refused subsets
6. Aggregate all-loan features (all-time + time windows)
7. Aggregate refused-loan features (all-time + time windows)
8. Merge & save final aggregated parquet

## 1. Imports & Configuration

In [1]:
import os
import pandas as pd
import numpy as np

DATA_ROOT = os.environ.get("HOMECREDIT_DATA_ROOT", r"a:\Dominic\My Projects\Home Credit\data")

PATH_APP_TRAIN     = os.path.join(DATA_ROOT, "Input", "application_train.csv")
PATH_PREV_LOANS    = os.path.join(DATA_ROOT, "Input", "previous_application.csv")
PATH_OUT_ENRICHED  = os.path.join(DATA_ROOT, "Staging",
                                   "prev_loan_data_processed.parquet")
PATH_OUT_AGG       = os.path.join(DATA_ROOT, "Staging",
                                   "prev_loans_aggregate_final_processed.parquet")

## 2. Load Data

In [2]:
# FIX: load only SK_ID_CURR from base — TARGET must NOT enter PrevLoans
# (original merged base with TARGET, causing it to leak into the saved parquet)
base = pd.read_csv(PATH_APP_TRAIN, usecols=['SK_ID_CURR'])
base = base.astype({'SK_ID_CURR': 'string'})

PrevLoans = pd.read_csv(PATH_PREV_LOANS)
PrevLoans = PrevLoans.astype({'SK_ID_CURR': 'string', 'SK_ID_PREV': 'string'})

# Restrict to training customers
PrevLoans = PrevLoans.merge(base, on='SK_ID_CURR', how='inner')

# FIX: sort AFTER merge — merge does not guarantee row order
PrevLoans.sort_values(
    by=['SK_ID_CURR', 'DAYS_DECISION'],
    ascending=[True, False],
    inplace=True,
    ignore_index=True,
)

print(f"PrevLoans shape: {PrevLoans.shape}")

PrevLoans shape: (1413701, 37)


## 3. Clean Invalid Sentinel Values

### 3a. Numeric sentinel `365243`

> **Fix:** the original hardcoded only 5 `DAYS_*` columns. We scan all numeric
> columns programmatically to catch any additional hits.

In [ ]:
SENTINEL = 365243
numeric_cols = PrevLoans.select_dtypes(include='number').columns
sentinel_cols = [col for col in numeric_cols if (PrevLoans[col] == SENTINEL).any()]
print(f"Columns containing sentinel {SENTINEL}: {sentinel_cols}")

for col in sentinel_cols:
    PrevLoans.loc[PrevLoans[col] == SENTINEL, col] = np.nan

print("Sentinel values replaced with NaN.")

### 3b. String sentinel `'XNA'`

`NAME_PAYMENT_TYPE` is the known carrier of `XNA` in this dataset.
We scan all object columns to catch any others.

In [ ]:
object_cols = PrevLoans.select_dtypes(include='object').columns
xna_cols = [col for col in object_cols if (PrevLoans[col] == 'XNA').any()]
print(f"Columns containing 'XNA': {xna_cols}")

for col in xna_cols:
    PrevLoans.loc[PrevLoans[col] == 'XNA', col] = np.nan

print("XNA values replaced with NaN.")

## 4. Derive Ratio Features

- **`CREDIT_TO_APPL`** — how much credit was approved relative to what was applied for.
  Values > 1 indicate the approved amount exceeded the application (e.g. top-up).
- **`GOODSPRICE_TO_AMT`** — goods price as a fraction of the applied amount.
- **`AMT_ANNUITY_TO_CREDIT_RATIO`** — annual repayment burden relative to loan size.

In [ ]:
PrevLoans['CREDIT_TO_APPL'] = np.where(
    PrevLoans['AMT_APPLICATION'] > 0,
    PrevLoans['AMT_CREDIT'] / PrevLoans['AMT_APPLICATION'],
    np.nan,
)
PrevLoans['GOODSPRICE_TO_AMT'] = np.where(
    PrevLoans['AMT_APPLICATION'] > 0,
    PrevLoans['AMT_GOODS_PRICE'] / PrevLoans['AMT_APPLICATION'],
    np.nan,
)
PrevLoans['AMT_ANNUITY_TO_CREDIT_RATIO'] = np.where(
    PrevLoans['AMT_CREDIT'] > 0,
    PrevLoans['AMT_ANNUITY'] / PrevLoans['AMT_CREDIT'],
    np.nan,
)

print("Ratio features added.")

## 5. Save Enriched Loan-Level Parquet

This file is loaded as metadata by the CC, POS, and Payments notebooks.
It contains one row per previous loan (`SK_ID_PREV`) with the derived ratios above.

> **Fix:** `TARGET` is no longer present here (it was never loaded from `base`).

In [ ]:
PrevLoans.to_parquet(PATH_OUT_ENRICHED, index=False)
print(f"Saved enriched parquet → {PATH_OUT_ENRICHED}")
print(f"Columns: {PrevLoans.shape[1]} | Rows: {PrevLoans.shape[0]:,}")

## 6. Split into Approved & Refused Subsets

- **`ApprovedLoans`** — all non-Refused applications (Approved, Canceled, Unused offer).
  Canceled loans with `AMT_CREDIT > 0` are retained because the credit was offered
  even if the client withdrew.
- **`RefusedLoans`** — applications where the lender declined.

In [ ]:
ApprovedLoans = PrevLoans[PrevLoans['NAME_CONTRACT_STATUS'] != 'Refused'].copy()
RefusedLoans  = PrevLoans[PrevLoans['NAME_CONTRACT_STATUS'] == 'Refused'].copy()

print(f"ApprovedLoans : {len(ApprovedLoans):,} rows")
print(f"RefusedLoans  : {len(RefusedLoans):,} rows")

## 7. Helper Functions

`flag_merge()` replaces the repeated `lambda x: 1 if x == 'both' else 0`
pattern used across 8 merge operations in the original notebook.

`build_window_agg()` encapsulates the 4-window aggregation pattern that was
duplicated identically for both the all-loans and refused-loans blocks.

In [ ]:
def flag_merge(left, right, on, flag_col):
    """Left-merge and add a binary flag: 1 if matched, 0 if not."""
    merged = left.merge(right, on=on, how='left', indicator=True)
    merged[flag_col] = (merged['_merge'] == 'both').astype(int)
    return merged.drop(columns=['_merge'])


def build_window_agg(df, agg_cols, days_back, suffix, prefix):
    """
    Aggregate *agg_cols* over the most recent *days_back* days.

    Parameters
    ----------
    df        : DataFrame with DAYS_DECISION and SK_ID_CURR columns
    agg_cols  : list of columns to aggregate
    days_back : int (negative), e.g. -365*3 for last 3 years
    suffix    : str appended after col name, e.g. '_LAST3Y'
    prefix    : str prepended to col name, e.g. 'PREVIOUS_TOTAL_LOAN_'
    """
    window = df[df['DAYS_DECISION'] >= days_back]
    agg = (
        window.groupby('SK_ID_CURR')[agg_cols]
        .agg(['mean'])
        .reset_index()
    )
    # Flatten MultiIndex; SK_ID_CURR has empty second level after reset_index
    agg.columns = [
        f"{prefix}{col[0]}_{col[1]}{suffix}".upper() if col[1] else col[0]
        for col in agg.columns
    ]
    return agg

## 8. All-Loan Aggregations

Customer-level summary across all previous applications (approved, canceled, and refused).

> **Fix 1:** `PREVIOUS_TOTAL_LOAN_COUNT` originally used
> `lambda x: sum(x == 'Refused')` — counting only Refused rows despite the 'total' name.
> Corrected to `'count'`.

> **Fix 2:** `PREVIOUS_TOTAL_LOAN_LAST_DECISION_DAY` replaces the misleadingly named
> `PREVIOUS_TOTAL_LOAN_REFUSED_DAY`. The max of `DAYS_DECISION` on all loans gives
> the most recent application date — not the last refusal. The refused-specific
> metric is computed properly in Section 9.

> **Fix 3:** `reset_index()` added — `SK_ID_CURR` was previously left in the index,
> causing a `KeyError` on the downstream merge.

In [ ]:
agg_cols = ['AMT_CREDIT', 'GOODSPRICE_TO_AMT', 'CREDIT_TO_APPL', 'AMT_ANNUITY_TO_CREDIT_RATIO']

PrevLoans_Aggregate_CurrentID = (
    PrevLoans.groupby('SK_ID_CURR').agg(
        # FIX: was lambda x: sum(x=='Refused') — now correctly counts all loans
        PREVIOUS_TOTAL_LOAN_COUNT=('NAME_CONTRACT_STATUS', 'count'),
        PREVIOUS_TOTAL_LOAN_AVG_LOAN_AMOUNT=('AMT_CREDIT', 'mean'),
        PREVIOUS_TOTAL_LOAN_AVG_GOODS_TO_AMT_RATIO=('GOODSPRICE_TO_AMT', 'mean'),
        PREVIOUS_TOTAL_LOAN_LAST3YEAR_COUNT=('DAYS_DECISION', lambda x: (x >= -365 * 3).sum()),
        PREVIOUS_TOTAL_LOAN_LASTYEAR_COUNT=('DAYS_DECISION', lambda x: (x >= -365).sum()),
        PREVIOUS_TOTAL_LOAN_LAST6MONTHS_COUNT=('DAYS_DECISION', lambda x: (x >= -180).sum()),
        PREVIOUS_TOTAL_LOAN_LAST3MONTHS_COUNT=('DAYS_DECISION', lambda x: (x >= -90).sum()),
        # FIX: renamed from REFUSED_DAY → LAST_DECISION_DAY (max on ALL loans, not refused)
        PREVIOUS_TOTAL_LOAN_LAST_DECISION_DAY=('DAYS_DECISION', 'max'),
        PREVIOUS_TOTAL_LOAN_MULTIPLE_ATTEMPTS_COUNT=(
            'FLAG_LAST_APPL_PER_CONTRACT', lambda x: (x == 'N').sum()
        ),
    )
    .reset_index()  # FIX: SK_ID_CURR was left in index without this
)

# Time-window mean aggregations
PrevLoans_3Y_Agg = build_window_agg(PrevLoans, agg_cols, -365*3, '_LAST3Y', 'PREVIOUS_TOTAL_LOAN_')
PrevLoans_1Y_Agg = build_window_agg(PrevLoans, agg_cols, -365,   '_LAST1Y', 'PREVIOUS_TOTAL_LOAN_')
PrevLoans_6M_Agg = build_window_agg(PrevLoans, agg_cols, -180,   '_LAST6M', 'PREVIOUS_TOTAL_LOAN_')
PrevLoans_3M_Agg = build_window_agg(PrevLoans, agg_cols, -90,    '_LAST3M', 'PREVIOUS_TOTAL_LOAN_')

print(f"All-loan base agg : {PrevLoans_Aggregate_CurrentID.shape}")
print(f"3Y / 1Y / 6M / 3M: {PrevLoans_3Y_Agg.shape} / {PrevLoans_1Y_Agg.shape} "
      f"/ {PrevLoans_6M_Agg.shape} / {PrevLoans_3M_Agg.shape}")

In [ ]:
PrevLoans_Aggregate_final = PrevLoans_Aggregate_CurrentID.copy()

PrevLoans_Aggregate_final = flag_merge(
    PrevLoans_Aggregate_final, PrevLoans_3Y_Agg, 'SK_ID_CURR', 'PREVIOUS_LOAN_IS_IN_LAST_3Y'
)
PrevLoans_Aggregate_final = flag_merge(
    PrevLoans_Aggregate_final, PrevLoans_1Y_Agg, 'SK_ID_CURR', 'PREVIOUS_LOAN_IS_IN_LAST_1Y'
)
PrevLoans_Aggregate_final = flag_merge(
    PrevLoans_Aggregate_final, PrevLoans_6M_Agg, 'SK_ID_CURR', 'PREVIOUS_LOAN_IS_IN_LAST_6M'
)
PrevLoans_Aggregate_final = flag_merge(
    PrevLoans_Aggregate_final, PrevLoans_3M_Agg, 'SK_ID_CURR', 'PREVIOUS_LOAN_IS_IN_LAST_3M'
)

del PrevLoans_Aggregate_CurrentID, PrevLoans_3Y_Agg, PrevLoans_1Y_Agg, PrevLoans_6M_Agg, PrevLoans_3M_Agg
print(f"All-loan aggregate shape: {PrevLoans_Aggregate_final.shape}")

## 9. Refused-Loan Aggregations

Customer-level summary across refused applications only.

> **Fix 1:** `PREVIOUS_REFUSED_LOAN_COUNT` used `lambda x: sum(x == 'Refused')`
> on a DataFrame already filtered to refused rows — every row matched, making the
> lambda equivalent to `len(x)`. Replaced with `'count'` for clarity.

> **Fix 2:** `reset_index()` added.

> **Fix 3:** `PREVIOUS_REFUSED_LOAN_REFUSED_DAY` now correctly uses `max(DAYS_DECISION)`
> on `RefusedLoans` (not all loans as in the original `PREVIOUS_TOTAL_LOAN_REFUSED_DAY`).

In [ ]:
RefusedLoans_Aggregate_CurrentID = (
    RefusedLoans.groupby('SK_ID_CURR').agg(
        # FIX: lambda was redundant on pre-filtered df; 'count' is equivalent and clearer
        PREVIOUS_REFUSED_LOAN_COUNT=('NAME_CONTRACT_STATUS', 'count'),
        PREVIOUS_REFUSED_LOAN_AVG_LOAN_AMOUNT=('AMT_CREDIT', 'mean'),
        PREVIOUS_REFUSED_LOAN_AVG_GOODS_TO_AMT_RATIO=('GOODSPRICE_TO_AMT', 'mean'),
        PREVIOUS_REFUSED_LOAN_LAST3YEAR_COUNT=('DAYS_DECISION', lambda x: (x >= -365 * 3).sum()),
        PREVIOUS_REFUSED_LOAN_LASTYEAR_COUNT=('DAYS_DECISION', lambda x: (x >= -365).sum()),
        PREVIOUS_REFUSED_LOAN_LAST6MONTHS_COUNT=('DAYS_DECISION', lambda x: (x >= -180).sum()),
        PREVIOUS_REFUSED_LOAN_LAST3MONTHS_COUNT=('DAYS_DECISION', lambda x: (x >= -90).sum()),
        # FIX: now correctly on RefusedLoans — most recent refusal date
        PREVIOUS_REFUSED_LOAN_LAST_REFUSED_DAY=('DAYS_DECISION', 'max'),
        PREVIOUS_REFUSED_LOAN_MULTIPLE_ATTEMPTS_COUNT=(
            'FLAG_LAST_APPL_PER_CONTRACT', lambda x: (x == 'N').sum()
        ),
    )
    .reset_index()  # FIX: SK_ID_CURR was left in index without this
)

RefusedLoans_3Y_Agg = build_window_agg(RefusedLoans, agg_cols, -365*3, '_LAST3Y', 'PREVIOUS_REFUSED_LOAN_')
RefusedLoans_1Y_Agg = build_window_agg(RefusedLoans, agg_cols, -365,   '_LAST1Y', 'PREVIOUS_REFUSED_LOAN_')
RefusedLoans_6M_Agg = build_window_agg(RefusedLoans, agg_cols, -180,   '_LAST6M', 'PREVIOUS_REFUSED_LOAN_')
RefusedLoans_3M_Agg = build_window_agg(RefusedLoans, agg_cols, -90,    '_LAST3M', 'PREVIOUS_REFUSED_LOAN_')

print(f"Refused-loan base agg: {RefusedLoans_Aggregate_CurrentID.shape}")

In [ ]:
RefusedLoans_Aggregate_final = RefusedLoans_Aggregate_CurrentID.copy()

RefusedLoans_Aggregate_final = flag_merge(
    RefusedLoans_Aggregate_final, RefusedLoans_3Y_Agg, 'SK_ID_CURR', 'PREVIOUS_REFUSED_LOAN_IS_IN_LAST_3Y'
)
RefusedLoans_Aggregate_final = flag_merge(
    RefusedLoans_Aggregate_final, RefusedLoans_1Y_Agg, 'SK_ID_CURR', 'PREVIOUS_REFUSED_LOAN_IS_IN_LAST_1Y'
)
RefusedLoans_Aggregate_final = flag_merge(
    RefusedLoans_Aggregate_final, RefusedLoans_6M_Agg, 'SK_ID_CURR', 'PREVIOUS_REFUSED_LOAN_IS_IN_LAST_6M'
)
RefusedLoans_Aggregate_final = flag_merge(
    RefusedLoans_Aggregate_final, RefusedLoans_3M_Agg, 'SK_ID_CURR', 'PREVIOUS_REFUSED_LOAN_IS_IN_LAST_3M'
)

del RefusedLoans_Aggregate_CurrentID, RefusedLoans_3Y_Agg, RefusedLoans_1Y_Agg, RefusedLoans_6M_Agg, RefusedLoans_3M_Agg
print(f"Refused-loan aggregate shape: {RefusedLoans_Aggregate_final.shape}")

## 10. Merge All-Loan & Refused-Loan Aggregates

Left-join refused-loan features onto the all-loan base.
Customers with no refusals receive `NaN` in all refused-loan columns.

In [ ]:
PrevLoans_Aggregate_final = PrevLoans_Aggregate_final.merge(
    RefusedLoans_Aggregate_final, on='SK_ID_CURR', how='left'
)

del RefusedLoans_Aggregate_final, ApprovedLoans, RefusedLoans
import gc; gc.collect()

print(f"Final aggregate shape: {PrevLoans_Aggregate_final.shape}")
PrevLoans_Aggregate_final.head(3)

## 11. Quality Checks

In [ ]:
# No duplicate customers
n_dupes = PrevLoans_Aggregate_final['SK_ID_CURR'].duplicated().sum()
assert n_dupes == 0, f"Duplicate SK_ID_CURR found: {n_dupes}"
print("No duplicate SK_ID_CURR.")

# TARGET must not be present
assert 'TARGET' not in PrevLoans_Aggregate_final.columns, "TARGET leaked into output!"
print("TARGET column not present — no data leakage.")

# Null rates
null_rates = PrevLoans_Aggregate_final.isna().mean().sort_values(ascending=False)
print(f"\nNull rate (top 10 columns):")
print(null_rates.head(10).to_string())

print(f"\nFinal row count: {len(PrevLoans_Aggregate_final):,}")

## 12. Save Aggregated Output

In [ ]:
PrevLoans_Aggregate_final.to_parquet(PATH_OUT_AGG, index=False)
print(f"Saved → {PATH_OUT_AGG}")

## 13. Final Cleanup

In [ ]:
del PrevLoans, PrevLoans_Aggregate_final, base
import gc; gc.collect()
print("All DataFrames released.")